# QuantBattle · predicciones de retornos diarios
Protocolo daily_return_mae_v1. Descarga train.parquet, validation.parquet y live.parquet de **la misma ronda y versión**. Este ejemplo entrena fuera de QuantBattle y produce el CSV que debes subir.
0.025 significa +2,5 %, −0.02 significa −2 %. No envíes probabilidades ni precios. El objetivo es el retorno completo del día siguiente al cierre de envíos. Las etiquetas históricas están redondeadas a pasos de 0.005; los resultados futuros se evalúan con retornos reales sin redondear. No se promete ventaja predictiva ni rentabilidad.

In [ ]:
%pip install pandas pyarrow scikit-learn numpy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

train = pd.read_parquet('train.parquet')
validation = pd.read_parquet('validation.parquet')
live = pd.read_parquet('live.parquet')
features = [c for c in train if c.startswith('feature_')]
assert features and set(features).issubset(validation) and set(features).issubset(live)
assert 'target' not in live
assert live.id.is_unique
print({'train': len(train), 'validation': len(validation), 'live': len(live)})

## Elegir usando validation
Se comparan dos configuraciones y una referencia que predice cero. La elección no usa el futuro live. Si cero obtiene menor error, también es un envío válido. Los errores de validation usan etiquetas históricas redondeadas y no son el resultado del torneo.

In [ ]:
seed = 20261026
candidates = []
for leaves in (7, 15):
    model = HistGradientBoostingRegressor(loss='absolute_error', max_iter=100, max_leaf_nodes=leaves, learning_rate=0.05, random_state=seed)
    model.fit(train[features], train.target)
    mae = mean_absolute_error(validation.target, model.predict(validation[features]))
    candidates.append((mae, leaves, model))
zero_mae = mean_absolute_error(validation.target, np.zeros(len(validation)))
best_mae, leaves, best_model = min(candidates, key=lambda item: item[0])
use_zero = zero_mae <= best_mae
print({'model_validation_mae': best_mae, 'zero_validation_mae': zero_mae, 'selected_zero': use_zero})

In [ ]:
prediction = np.zeros(len(live)) if use_zero else best_model.predict(live[features])
prediction = np.clip(prediction, -1, 10)
assert np.isfinite(prediction).all()
submission = pd.DataFrame({'id': live.id.astype(str), 'prediction': prediction})
assert submission.id.is_unique and len(submission) == len(live)
submission.to_csv('predicciones_retornos.csv', index=False)
print(submission.head())
print('Sube predicciones_retornos.csv a la ronda exacta de la que descargaste live.')

## Registrar y conservar evidencia
En QuantBattle indica el nombre del modelo realmente seleccionado (incluye si fue la referencia cero), sus parámetros y la semilla. Conserva la versión de la ronda, el CSV y el comprobante SHA-256. El último archivo válido antes del cierre compite. Se requieren cuatro rondas válidas para el ranking general; scores, símbolos y precios reales se revelan juntos al terminar el torneo semanal.
Los datos fuente son de Binance Vision y los derivados se ofrecen para investigación no comercial bajo CC BY-NC-SA 4.0 y las condiciones de Binance Public Data. La ofuscación de variables no garantiza anonimato criptográfico.